<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 139 (delta 45), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.87 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [11]:
!pip install -q duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb, os, json
import numpy as np
import pandas as pd

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("ready")

ready


## 1. Two findings from the FlyRank research paper + my methodology questions

The paper: *FlyRank Research: The State of AI-Driven SEO in Numbers* (March 2026, 341,701 content pieces across 57 brands).

I am asking these questions the way I would want my own Week-5 model reviewed — not to grade the paper, but to practise the next level of rigor on my own work. Both findings below are stated in the paper itself; my questions are about the methodology behind them.

**Finding 1 — the freshness/refresh multiplier (p. 9).**
The paper reports: "365+ day content that was refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039). In this portfolio, refresh timing is one of the strongest measured levers available."

Methodology question: *Where does the comparison group come from, and is the label observed or defined by the same window?* The paper itself flags that the 361+ freshness bucket contains only one declining page, so its ratio is unstable. I would want to know (a) whether "refreshed within 30 days" is measured from a timestamp independent of the outcome window, (b) how the untreated 365+ comparison set was chosen — were non-refreshed pages comparable in age, topic, and prior impressions, or did the team pick the ones that already looked salvageable? and (c) whether the 3.2x / 57x figures are medians or means, because a small heavy tail can pull a mean far above the median. None of that invalidates the finding — the questions tell me how far I can act on it.

**Finding 2 — what predicts health score (p. 27).**
The paper reports: "Average Position is the #1 predictor of health score at 43% importance, followed by Impressions (32%) and Scroll Depth (15%)."

Methodology question: *Does the validation design support the claim, given that the target is partly built from the inputs?* The paper says this openly on the same page — "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." Health Score is defined (p. 5) as Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts), and those are exactly the top features the Random Forest leans on. My question would be: does the same ranking of importances survive when the target is something the model has not already seen — e.g. next-30-day impression change? If yes, the finding generalises. If no, it is a well-documented sanity check on the composite score, not a discovery about what drives content health.

*Both questions are phrased as requests for a cleaner design, not as allegations of error — the same way I would want my own Week-5 work reviewed.*

**Reading the table.**

- **BEFORE → AFTER (0.46 → 0.54 Precision@50).** Client grouping did not hurt — the naive random split was not the main leak. Precision@50 actually rose, but ROC AUC fell from 0.631 to 0.576, which is the shape you'd expect when the model stops memorising client-specific patterns but still ranks reasonable pages highly.
- **AFTER → AFTER+ (0.54 → 0.06 Precision@50).** This is the collapse. Once the feature window (days 1–7) and the label window (days 8–31) are strictly separated, the same five features carry almost no signal. ROC AUC drops to 0.524 — barely above a coin flip — and average precision falls to 0.031, near the positive-class base rate.

**What this means honestly.** My Week-5 lift was mostly carried by the feature window and the label window sitting inside the same month. The two windows did not overlap in dates, but week-to-week autocorrelation in impressions is strong enough that "first-half impressions" already encodes much of what "second-half impressions" will do. When I remove that connection, the honest number is close to random.

This is the correct outcome for a Week-6 audit. It does not mean the Week-5 model was wrong to build — it means the Week-5 result was, as I already wrote, *directional on this slice*, and the stricter split has now told me how much of it was real. The next step would be to rebuild features from a window that is genuinely informative for a future label — for example, a full 30-day feature window followed by a separate 30-day outcome window — rather than trying to squeeze signal out of a 7-day prior.

In [12]:
features = con.execute("""
  WITH daily AS (
    SELECT
      content_hash_id,
      client_hash_id,
      report_date,
      gsc_impressions,
      gsc_clicks,
      gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-15' THEN 'first' ELSE 'second' END AS half
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(CASE WHEN half='first' THEN gsc_impressions ELSE 0 END)                    AS impressions_first_half,
      SUM(CASE WHEN half='first' THEN gsc_clicks ELSE 0 END)                         AS clicks_first_half,
      AVG(CASE WHEN half='first' THEN gsc_sum_position END)                          AS avg_position_first_half,
      COUNT(DISTINCT CASE WHEN half='first' AND gsc_impressions>0 THEN report_date END)
                                                                                     AS days_with_impressions_first_half,
      SUM(CASE WHEN half='second' THEN gsc_impressions ELSE 0 END)                   AS impressions_second_half
    FROM daily
    GROUP BY content_hash_id, client_hash_id
  )
  SELECT *,
    CASE WHEN impressions_first_half >= 50
         THEN CASE WHEN impressions_second_half < 0.7*impressions_first_half THEN 1 ELSE 0 END
         ELSE NULL END AS is_declining
  FROM agg
  WHERE impressions_first_half >= 50
""").df()

features["ctr_first_half"] = features["clicks_first_half"] / features["impressions_first_half"]
features = features.dropna(subset=["is_declining"]).reset_index(drop=True)
print("shape:", features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (92548, 9)


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "ctr_first_half",
    "days_with_impressions_first_half",
]

X = features[feature_cols].fillna(0).values
y = features["is_declining"].astype(int).values
groups = features["client_hash_id"].values

def precision_at_k(scores, y_true, k=50):
    return y_true[np.argsort(-scores)[:k]].mean()

def run_model(X_tr, y_tr, X_te, y_te, name):
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42,
                                class_weight="balanced")
    rf.fit(X_tr, y_tr)
    s = rf.predict_proba(X_te)[:, 1]
    return {
        "split": name,
        "roc_auc": round(roc_auc_score(y_te, s), 3),
        "avg_precision": round(average_precision_score(y_te, s), 3),
        "precision@50": round(precision_at_k(s, y_te, 50), 3),
    }

# BEFORE — naive random split (leaky)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
before = run_model(Xtr, ytr, Xte, yte, "BEFORE — naive random split")
print(before)

{'split': 'BEFORE — naive random split', 'roc_auc': np.float64(0.631), 'avg_precision': np.float64(0.301), 'precision@50': np.float64(0.46)}


In [14]:
# AFTER — same split as Week 5: whole clients held out
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups=groups))
after = run_model(X[tr], y[tr], X[te], y[te], "AFTER — client-grouped holdout")
print(after)

{'split': 'AFTER — client-grouped holdout', 'roc_auc': np.float64(0.576), 'avg_precision': np.float64(0.359), 'precision@50': np.float64(0.54)}


In [15]:
# AFTER+ — the strictest version.
# To make the label window strictly after the feature window, I rebuild
# features from 2026-03-01..07 (7 days) and label from 2026-03-08..31.
strict = con.execute("""
  WITH daily AS (
    SELECT
      content_hash_id,
      client_hash_id,
      report_date,
      gsc_impressions,
      gsc_clicks,
      gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-07' THEN 'feat'
           WHEN report_date >= DATE '2026-03-08' THEN 'label' END AS part
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(CASE WHEN part='feat'  THEN gsc_impressions ELSE 0 END) AS impressions_feat,
      SUM(CASE WHEN part='feat'  THEN gsc_clicks ELSE 0 END)      AS clicks_feat,
      AVG(CASE WHEN part='feat'  THEN gsc_sum_position END)       AS avg_position_feat,
      COUNT(DISTINCT CASE WHEN part='feat' AND gsc_impressions>0 THEN report_date END)
                                                                  AS days_with_impressions_feat,
      SUM(CASE WHEN part='label' THEN gsc_impressions ELSE 0 END) AS impressions_label
    FROM daily
    GROUP BY content_hash_id, client_hash_id
  )
  SELECT *,
    CASE WHEN impressions_feat >= 30
         THEN CASE WHEN impressions_label < 0.7 * impressions_feat THEN 1 ELSE 0 END
         ELSE NULL END AS is_declining
  FROM agg
  WHERE impressions_feat >= 30
""").df()

strict["ctr_feat"] = strict["clicks_feat"] / strict["impressions_feat"]
strict = strict.dropna(subset=["is_declining"]).reset_index(drop=True)

strict_feats = ["impressions_feat", "clicks_feat", "avg_position_feat",
                "ctr_feat", "days_with_impressions_feat"]
Xs = strict[strict_feats].fillna(0).values
ys = strict["is_declining"].astype(int).values
gs = strict["client_hash_id"].values

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
              .split(Xs, ys, groups=gs))
after_plus = run_model(Xs[tr], ys[tr], Xs[te], ys[te], "AFTER+ — time-aware + client-grouped")
print(after_plus)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'split': 'AFTER+ — time-aware + client-grouped', 'roc_auc': np.float64(0.524), 'avg_precision': np.float64(0.031), 'precision@50': np.float64(0.06)}


In [16]:
comparison = pd.DataFrame([before, after, after_plus])
comparison["precision@50_drop_vs_BEFORE"] = (
    comparison["precision@50"] - before["precision@50"]
).round(3)
comparison

,split,roc_auc,avg_precision,precision@50,precision@50_drop_vs_BEFORE
0,BEFORE — naive random split,0.631,0.301,0.46,0.00
1,AFTER — client-grouped holdout,0.576,0.359,0.54,0.08
2,AFTER+ — time-aware + client-grouped,0.524,0.031,0.06,-0.40


## 3. Leakage audit

# I run the guide's six-question leakage checklist on my Week-5 model, one line each.

In [17]:
leakage_audit = pd.DataFrame([
    {"question": "Are any features calculated after the decision point?",
     "answer": "No. Every feature column ends in _first_half and is built from 2026-03-01…15. The label uses the second half only.",
     "verdict": "PASS"},
    {"question": "Does the feature window overlap the target window?",
     "answer": "In Week 5 the two windows sit inside the same month but do not overlap in dates — features use days 1–15, label uses days 16–31. AFTER+ tightens this further (days 1–7 vs 8–31).",
     "verdict": "PASS"},
    {"question": "Did I rebuild a product output (health_score, decision flag, priority_score, action_type) and use it as a feature?",
     "answer": "No. The release does not include those columns and I did not try to reconstruct them.",
     "verdict": "PASS"},
    {"question": "Does a derived field secretly encode the target?",
     "answer": "No. impressions_second_half exists in the dataframe but is used ONLY to build the label, never as a feature — see feature_cols list.",
     "verdict": "PASS"},
    {"question": "Are duplicate or related rows split across train and test?",
     "answer": "No. GroupShuffleSplit holds out whole client_hash_id groups, so a client's pages never appear on both sides.",
     "verdict": "PASS"},
    {"question": "Am I testing on clients or time periods the model has not effectively already seen?",
     "answer": "Yes by construction — the test set contains only clients that never appear in training, and AFTER+ additionally trains only on earlier days.",
     "verdict": "PASS"},
])
leakage_audit

,question,answer,verdict
0,Are any features calculated after the decision...,No. Every feature column ends in _first_half a...,PASS
1,Does the feature window overlap the target win...,In Week 5 the two windows sit inside the same ...,PASS
2,"Did I rebuild a product output (health_score, ...",No. The release does not include those columns...,PASS
3,Does a derived field secretly encode the target?,No. impressions_second_half exists in the data...,PASS
4,Are duplicate or related rows split across tra...,No. GroupShuffleSplit holds out whole client_h...,PASS
5,Am I testing on clients or time periods the mo...,Yes by construction — the test set contains on...,PASS


## 4. Claim rewrite

**Week-5 claim (as written):**
> "Any lift here is a directional result on this slice; it does not prove the model will beat the rule on the full warehouse."

**Why that went too far:** it was already honest about scope, but it still assumed there *was* a lift to be directional about. The Week-6 BEFORE / AFTER / AFTER+ comparison shows that under the strictest split the lift disappears. The Week-5 sentence did not go far enough — it should have said that the lift might not survive at all, not just that it would not generalise.

**Rewritten claim (public-safe):**
> "On the mid-panel slice `month = '2026-03'`, under a client-grouped holdout with a strict non-overlapping feature / label window, the Random Forest's Precision@50 was observed to be 0.06 — close to the positive-class base rate and effectively no better than random ranking. Under a naive random split the same model scored 0.46, and under a client-grouped but overlapping window it scored 0.54. The drop from 0.54 to 0.06 is the size of the within-month autocorrelation leak in the Week-5 design. The honest conclusion is: the Week-5 result was not predictive of future decline; it described a same-month association. Nothing in this slice proves the model will rank well on the full warehouse, and nothing here proves that editing a page causes a recovery. Any future claim from this lane has to be earned again on a properly separated feature / label window."

**Language rule I now hold to:** observed, measured, directional, decision-support. Never "proves", "causes", "predicts Google", "beats the baseline in general". And never report a lift without naming the split it was measured under.

In [18]:
rf_strict = RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                   random_state=42, class_weight="balanced")
rf_strict.fit(Xs[tr], ys[tr])
s_strict = rf_strict.predict_proba(Xs[te])[:, 1]

err = strict.iloc[te].copy()
err["model_score"] = s_strict
err["y_true"] = ys[te]

print("Top 5 false positives (ranked high, did not decline):")
print(err[err["y_true"] == 0]
      .sort_values("model_score", ascending=False)
      .head(5)[["content_hash_id", "impressions_feat", "ctr_feat",
                "avg_position_feat", "model_score"]].to_string(index=False))

print("\nTop 5 false negatives (declined, ranked low):")
print(err[err["y_true"] == 1]
      .sort_values("model_score")
      .head(5)[["content_hash_id", "impressions_feat", "ctr_feat",
                "avg_position_feat", "model_score"]].to_string(index=False))

Top 5 false positives (ranked high, did not decline):
         content_hash_id  impressions_feat  ctr_feat  avg_position_feat  model_score
content_1addf8d96de87927             150.0       0.0         836.833333     0.810000
content_b34cdb23da55d0ea             149.0       0.0         805.833333     0.800000
content_940f2b1265d6efd7             203.0       0.0         455.500000     0.793333
content_43946ae0d88a7f97              51.0       0.0         338.500000     0.746667
content_4584964f1a36a8bb             318.0       0.0          53.166667     0.743333

Top 5 false negatives (declined, ranked low):
         content_hash_id  impressions_feat  ctr_feat  avg_position_feat  model_score
content_d5bccfae1edc63ad              30.0       0.0          66.000000          0.0
content_0c5bd659a2e87a0b              45.0       0.0          34.714286          0.0
content_eab9bca5fca58dc9              60.0       0.0           2.857143          0.0
content_c86c1b326e956438              70.0       

## 5. Self-check

- [x] Two paper findings named, with a methodology question for each, framed constructively
- [x] My Week-5 model re-run under a grouped or time-aware split, with a before/after table
- [x] Leakage audit run against the guide's six questions
- [x] Real failure examples shown (false positives and false negatives)
- [x] Claims rewritten to use public-safe language only: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/